<a href="https://colab.research.google.com/github/EofK/benchmark_ml_and_dl_models_with_tabular_datasets/blob/master/benchmark_12_dl_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Benchmark 12 DL models on multiple datasets on a supervised binary classification task.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


If using Google Drive, the paths should start with /content/drive/MyDrive/.

In [ ]:
# ==============================================================================
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
# ==============================================================================
# INSTRUCTIONS: Change this path to point to your main project folder.
USER_PATH = '/content/drive/MyDrive/ml_dl_benchmark_results'

# The code below automatically builds your file paths based on the USER_PATH
RESULTS_PATH = f'{USER_PATH}/ml_dl_benchmark_results_brier.csv'
ERRORS_PATH  = f'{USER_PATH}/benchmark_errors.csv'

DATASET_DIR = Path(f'{USER_PATH}/ml_dl_datasets/')
BASE        = Path(f'{USER_PATH}/Primary_Excel_Files')

# Verify the paths are set correctly
print(f"Datasets will be loaded from: {DATASET_DIR}")
print(f"Results will be saved to: {RESULTS_PATH}")

In [ ]:
# Verify that Pytorch can see GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
# Installs for timezone handling, Excel writing, and dataset complexity measurement
!pip install pytz
!pip install openpyxl
!pip install problexity -q

In [ ]:
# Confirm GPU
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
# Install torchinfo library
!pip install torchinfo -q

In [ ]:
# Install a better option than individual packages. It includes NODE, FT-Transformer, TabNet, and several other tabular models under one roof.

!pip install pytorch-tabular torchinfo -q

In [ ]:
# Verify pytorch-tabular installed and what models are available.
from pytorch_tabular import available_models
models = available_models()
for i, model in enumerate(models):
    print(model, end='\n' if (i + 1) % 4 == 0 else ',  ')

In [ ]:
# Workbook uses four widedeep models
!pip install pytorch-widedeep -q

In [ ]:
!pip install pytorch-tabular==1.2.0

Define helper functions to save benchmarking results and any errors incurred on a dataset.



In [ ]:
import pandas as pd
import os
import numpy as np
from sklearn.metrics import pairwise_distances
import csv # Import the csv module for quoting options
from datetime import date, datetime # Import date and datetime modules
import pytz # Import pytz for timezone handling
from pathlib import Path # Import Path for file paths
# Define the Pacific Timezone
pacific_tz = pytz.timezone('America/Los_Angeles')

def save_benchmark_results(dataset_name, n_samples, n_features, n_numeric_features,
                           n_categorical_features, model_name, accuracy, f1_weighted,
                           auc, brier_score, training_time, inference_time, pred_sec,
                           total_time, runtime_per_1000_samples, processor_type,
                           n_trainable_params, n_epochs_run, early_stopped,
                           gpu_memory_peak_mb, flops, mean_pred_proba, file_path):
    results_df = pd.DataFrame({
        'Dataset':                       [dataset_name],
        'N_Samples':                     [n_samples],
        'N_Features':                    [n_features],
        'N_Numeric_Features':            [n_numeric_features],
        'N_Categorical_Features':        [n_categorical_features],
        'Model':                         [model_name],
        'Accuracy':                      [accuracy],
        'F1_Weighted':                   [f1_weighted],
        'AUC':                           [auc],
        'Brier_Score':                   [brier_score],
        'Training_Time_seconds':         [training_time],
        'Inference_Time_seconds':        [inference_time],
        'Preds_per_Second':              [pred_sec],
        'Total_Runtime_seconds':         [total_time],
        'Runtime_per_1000_Samples':      [runtime_per_1000_samples],
        'Processor':                     [processor_type],
        'N_Trainable_Params':            [n_trainable_params],
        'N_Epochs_Run':                  [n_epochs_run],
        'Early_Stopped':                 [early_stopped],
        'GPU_Memory_Peak_MB':            [gpu_memory_peak_mb],
        'FLOPs':                         [flops],
        'Mean_Pred_Proba_Positive_Class':[mean_pred_proba],
    })
    if not os.path.exists(file_path):
        results_df.to_csv(file_path, mode='w', header=True, index=False,
                          float_format='%.4f', quoting=csv.QUOTE_NONNUMERIC)
        print(f"Created new benchmark results file: {file_path}")
    else:
        results_df.to_csv(file_path, mode='a', header=False, index=False,
                          float_format='%.4f', quoting=csv.QUOTE_NONNUMERIC)
        print(f"Appended results to: {file_path}")


def save_error(dataset_name, model_name, error_message, file_path=ERRORS_PATH):
    now_pacific = datetime.now(pytz.utc).astimezone(pacific_tz)
    error_df = pd.DataFrame({
        'Dataset':   [dataset_name],
        'Model':     [model_name],
        'Timestamp': [now_pacific.strftime('%b %d, %Y %I:%M:%S %p')],
        'Error':     [str(error_message)]
    })
    if not os.path.exists(file_path):
        error_df.to_csv(file_path, mode='w', header=True, index=False)
    else:
        error_df.to_csv(file_path, mode='a', header=False, index=False)


# Get the current UTC time and convert to Pacific Time
now_pacific = datetime.now(pytz.utc).astimezone(pacific_tz)
print("Defined save_benchmark_results and save_error functions.")
print("Defined k_1 and distributions function")
print(f"\nGlobal variables and functions defined on {now_pacific.strftime('%b %d, %Y %I:%M:%S %p')}")

In [ ]:
import pytorch_widedeep.models as wd_models

# Dynamically list and print the names of model classes that are importable
# based on previous successful imports.
# Exclude helper/base classes like WideDeep itself (as it orchestrates others)
# if the user wants individual components.

# For the purpose of listing 'unique DL models', list the main components.
# The names below correspond to classes that can be directly instantiated
# as deep components or the overall WideDeep orchestrator.
model_classes_to_list = [
    wd_models.WideDeep,
    wd_models.TabMlp,
    wd_models.TabFastFormer,
    wd_models.TabTransformer,
    wd_models.SAINT,
    wd_models.TabNet,
    wd_models.FTTransformer,
    wd_models.SelfAttentionMLP
]

print("Available pytorch_widedeep model classes (dynamically imported):")
for model_class in model_classes_to_list:
    print(f"- {model_class.__name__}")


In [ ]:
# Select datasets for DL model evaluation.

# The sole purpose of this code cell is to select the datasets to be used
# in subsequent code cells.  This code cell allows for a simple selection of
# which datasets will be used in the evaluation of the DL models tha ooccurs
# in the the two cells that follow this dataset selection cell.

# ── Master list of all 159 datasets ──────────────────────────────────────────
ALL_DATASETS = [
    'adult_income_clean_train_test_binarized.csv',        # 1
    'ailerons_binarized_P.csv',                           # 2
    'air_quality_and_pollution_assessment_binarized_HAZARDOUS.csv',  # 3
    'albert.csv',                                         # 4
    'Amazon_access_converted.csv',                        # 5
    'analcatdata_aids_converted_binarized_MALE.csv',      # 6
    'analcatdata_asbestos_converted_binarized_(3)_ABOVE_LEGAL_LIMIT.csv',  # 7
    'analcatdata_bankruptcy_converted.csv',               # 8
    'analcatdata_boxing1_converted_binarized_LEWIS.csv',  # 9
    'analcatdata_boxing2_converted_binarized_TRINIDAD.csv',  # 10
    'analcatdata_creditscore_converted.csv',              # 11
    'analcatdata_cyyoung8092_converted.csv',              # 12
    'analcatdata_cyyoung9302_converted.csv',              # 13
    'analcatdata_fraud_converted.csv',                    # 14
    'analcatdata_japansolvent_converted.csv',             # 15
    'analcatdata_lawsuit_converted.csv',                  # 16
    'appendicitis_converted_binarized_2.csv',             # 17
    'australian_converted.csv',                           # 18
    'backache_converted.csv',                             # 19
    'Bank Customer Churn Prediction.csv',                 # 20
    'bank_marketing_data_cleaned_binarized_yes.csv',      # 21
    'banknote_authentication.csv',                        # 22
    'BEED_binarized_1.csv',                               # 23
    'biomed_converted_binarized_CARRIER.csv',             # 24
    'bioresponse.csv',                                    # 25
    'blood_transfusion_classification_binarized_DONATED.csv',  # 26
    'BNG_breast-w_converted_binarized_malignant.csv',     # 27
    'BNG_cmc_converted_binarized_2.csv',                  # 28
    'breast_cancer_dataset.csv',                          # 29
    'breast_cancer_prediction_binarized_M.csv',           # 30
    'california_environment_conditions_cleaned.csv',      # 31
    'california_housing_classification.csv',              # 32
    'cardiovascular_disease_classification.csv',          # 33
    'churn.csv',                                          # 34
    'clean1_converted.csv',                               # 35
    'cleve_converted.csv',                                # 36
    'cmc_binarized_1.csv',                                # 37
    'codrna_cleaned_binarized_bracket_0 1.csv',           # 38
    'company_bankruptcy_prediction.csv',                  # 39
    'compass_binarized_0.csv',                            # 40
    'comprehensive _diabetes_clinical_binarized_1.csv',   # 41
    'contraceptive_method_choice_binarized_1.csv',        # 42
    'corral_converted.csv',                               # 43
    'covertype.csv',                                      # 44
    'credit.csv',                                         # 45
    'credit_approval_binarized_plus_sign.csv',            # 46
    'credit_g_binarized_BAD.csv',                         # 47
    "creditcard_binarized_1.csv",                         # 48
    'default_of_credit_card_clients.csv',                 # 49
    'diabetes_health_indicators.csv',                     # 50
    'diabetes130US.csv',                                  # 51
    'diabetic_retinopathy_debrecen.csv',                  # 52
    'digits_binarized_7.csv',                             # 53
    'e_commerce_shipping_data.csv',                       # 54
    'egg_eye_state_converted_binarized_1.csv',            # 55
    'electricity_binarized_UP.csv',                       # 56
    'elevators_binarized_P.csv',                          # 57
    'ember2024_50000.csv',                                # 58
    'fico_heloc_cleaned_binarized_Good.csv',              # 59
    'fitness_class_2212.csv',                             # 60
    'gallstone.csv',                                      # 61
    'gina_agnostic_binarized_1.csv',                      # 62
    'haberman_converted_binarized_2.csv',                 # 63
    'heart_disease_dataset_comprehensive.csv',            # 64
    'heart-statlog_converted_binarized_PRESENT.csv',      # 65
    'heloc.csv',                                          # 66
    'higgs_sample20000_cleaned.csv',                      # 67
    'hill_valley.csv',                                    # 68
    'house_16h_binarized_P.csv',                          # 69
    'HTRU2.csv',                                          # 70
    'in_vehicle_coupon_recommendation.csv',               # 71
    'insurance.csv',                                      # 72
    'Invistico_Airline_binarized_satisfied.csv',          # 73
    'ionosphere_converted_binarized_B.csv',               # 74
    'iranian_churn.csv',                                  # 75
    'iris_binarized.csv',                                 # 76
    'irish_converted_binarized_TAKEN.csv',                # 77
    'is_this_a_good_customer.csv',                        # 78
    'jannis.csv',                                         # 79
    'jasmine_converted.csv',                              # 80
    'jm1_binarized_TRUE.csv',                             # 81
    'kc1_converted_binarized_TRUE.csv',                   # 82
    'KDD_binarized_1.csv',                                # 83
    'kdd_ipums_la_97_small_binarized_P.csv',              # 84
    'kddcup99_binarized_sample25000.csv',                 # 85
    'liver-disorders_converted_binarized_1.csv',          # 86
    'lupus_converted_binarized_DEAD.csv',                 # 87
    'madeline_converted.csv',                             # 88
    'MagicTelescope_converted_binarized_g.csv',           # 89
    'make_blobs_dataset.csv',                             # 90
    'make_circles_dataset.csv',                           # 91
    'make_classification_binarized_4.csv',                # 92
    'make_moons_dataset.csv',                             # 93
    'mammography_converted_binarized_1.csv',              # 94
    'marketing_campaign_cleaned_dates_numeric.csv',       # 95
    'medical_appointment_no_shows_convert_dates_binarized_YES.csv',  # 96
    'Medical-Appointment.csv',                            # 97
    'miniboone_binarized_TRUE.csv',                       # 98
    'multiple_sclerosis_disease_cols_removed_binarized_1.csv',  # 99
    'mux6_converted.csv',                                 # 100
    'nasa_nearest_object_valued_binarized_TRUE.csv',      # 101
    'naticusdroid_android_permissions.csv',               # 102
    'nhanes_age_predictions_subset_binarized_Senior.csv', # 103
    'numerai28.6_converted.csv',                          # 104
    'occupancy_detection_combined_date_converted.csv',    # 105
    'online_shoppers_binarized_TRUE.csv',                 # 106
    'ozone-level-8hr_binarized_2.csv',                    # 107
    'page-blocks_binarized_P.csv',                        # 108
    'parity5_converted.csv',                              # 109
    'pc1_binarized_TRUE.csv',                             # 110
    'pc3_binarized_TRUE.csv',                             # 111
    'pc4_binarized_TRUE.csv',                             # 112
    'philippine_converted.csv',                           # 113
    'PhiUSIIL Phishing URL - Website_sample_25000.csv',   # 114
    'phoneme_binarized_1.csv',                            # 115
    'pima_indians_diabetes.csv',                          # 116
    'pol_binarized_P.csv',                                # 117
    'pollen_binarized_P.csv',                             # 118
    'postoperative-patient-data_converted_binarized_2.csv',  # 119
    'predict_students_dropout_and_academic_success_binarized_GRADUATE.csv',  # 120
    'prnn_crabs_converted_binarized_BLUE_FORM.csv',       # 121
    'prnn_synth_converted.csv',                           # 122
    'pumpkin_seeds_binarized_URGUP_SIVRISI.csv',          # 123
    'qsar_binarized_RB.csv',                              # 124
    'qsar-biodeg_binarized_YES.csv',                      # 125
    'Raisin_binarized_BESNI.csv',                         # 126
    'rice_cammeo_and_osmancik_binarized.csv',             # 127
    'ringnorm_binarized_1.csv',                           # 128
    'rl_binarized_1.csv',                                 # 129
    'road_safety.csv',                                    # 130
    'sa-heart_converted_binarized_2.csv',                 # 131
    'sandtander_customer_satisfaction_binarized_TRUE_sample25000.csv',  # 132
    'satellite_converted_binarized_Anomaly.csv',          # 133
    'segment_binarized_P.csv',                            # 134
    'sepsis_survival_minimal_clinical _records.csv',      # 135
    'shipping.csv',                                       # 136
    'shrutime_binarized_1.csv',                           # 137
    'sick_binarized_SICK.csv',                            # 138
    'smoking_drinking_dataset_Ver01_binarized_Y_sample100000.csv',  # 139
    'sonar_converted_binarized_ROCK.csv',                 # 140
    'spambase_binarized_1.csv',                           # 141
    'spect_converted.csv',                                # 142
    'spectf_converted.csv',                               # 143
    'stroke_prediction_dataset.csv',                      # 144
    'student_depression_dataset.csv',                     # 145
    'svmguide3_binarized_1.csv',                          # 146
    'sylvine_converted.csv',                              # 147
    'taiwanese_bankruptcy_prediction.csv',                # 148
    'TCGA_GBM_LGG_Mutations_all_binarized_LGG_fix_cat_features.csv',  # 149
    'telecom_customer_churn_binarized_YES.csv',           # 150
    'twonorm_converted_binarized_1.csv',                  # 151
    'vehicleNorm_cleaned_binarized_100 -1_bracket_sample25000.csv',  # 152
    'visualizing_soil_binarized_P.csv',                   # 153
    'vulnonevul.csv',                                     # 154
    'waterQuality1.csv',                                  # 155
    'waveform-5000_converted_binarized_P.csv',            # 156
    'wilt_seed_binarized_2.csv',                          # 157
    'wine_quality_binarized_6.csv',                       # 158
    'yeast_ml8_converted.csv',                            # 159
    'BAF_variant_III_cleaned_prepped.csv',                           # 160
    'BayesianNetworkGenerator_mushroom_binarized_E_cleaned_prepped.csv',  # 161
    'BNG_colic_binarized_YES_cleaned_prepped.csv',                   # 162
    'BNG_heart-statlog_binarized_PRESENT_cleaned_prepped.csv',       # 163
    'BNG_hepatitis_binarized_LIVE_cleaned_prepped.csv',              # 164
    'BNP_Paribas_Cardif_Claims_Management_cleaned_prepped.csv',      # 165
    'college_scorecard_cleaned_prepped.csv',                         # 166
    'Customer_Churn_Classification_cleaned_prepped.csv',             # 167
    'GiveMeSomeCredit_binarized_YES_cleaned_prepped.csv',            # 168
    'higgs_boson_2014_binarized_S_cleaned_prepped.csv',              # 169
    'LT-Vehicle-Loan-Default-Prediction_cleaned_prepped.csv',        # 170
    'porto-seguro_cleaned_prepped.csv',                              # 171
    'sf_police_incidents_binarized_YES_cleaned_prepped.csv',         # 172
    'online_payments_fraud_detection_sample_800000.csv',             # 173
    'Click_cleaned_prepped.csv',                                     # 174
    'BNG_labor_binarized_GOOD_cleaned_prepped.csv',                  # 175
    'BNG_kr_vs_kp_binarized_WON_cleaned_prepped.csv',                # 176
    'BNG_cylinder_bands_binarized_BAND_cleaned_prepped.csv',         # 177
    'BAF_base_cleaned_prepped.csv',                                  # 178
    'airlines_delay_cleaned_prepped.csv',                            # 179
    'ACSPublicCoverage_binarized_1_cleaned_prepped.csv',             # 180
    'ACSIncome_binarized_1_cleaned_sample_1000000_prepped.csv',      # 181
    'susy_sample_1000000_cleaned_prepped.csv',                       # 182
    'hepmass_all_train_sample1000000_cleaned_prepped.csv',           # 183
]

# ── Choose datasets by index range (1-based, inclusive) ──────────────────────
START = 1
END   = 2

DATASETS = ALL_DATASETS[START-1:END]

# Confirm selection
print(f"Selected datasets ({START} to {END}):")
for i, d in enumerate(DATASETS, start=START):
    print(f"  {i}. {d}")

##Loop through eight pytorch tabular models

## Data Preprocessing Fixes Applied to This Loop

- **Column name cleaning**: Special characters in column names are replaced with underscores to prevent model errors
- **Quote stripping**: Embedded single quotes are stripped from string column values
- **Target encoding**: LabelEncoder output is cast to `int32` to prevent torchmetrics multiclass misclassification bug
- **Categorical/numeric detection**: Columns are automatically classified as categorical or numeric based on dtype
- **NaN imputation**: Missing values in numeric columns are imputed using median strategy; categorical columns are not imputed
- **Float32 conversion**: Numeric columns are cast to `float32` as pytorch_tabular requires float, not int
- **Accuracy calculation**: Accuracy is calculated directly from `model.predict()` rather than `model.evaluate()`, which triggers a torchmetrics multiclass bug on certain datasets

In [ ]:
# Loop through eight DL models on multiple datasets.

# Tabular models were erroring out on integer values in two datasets
# This code cell replaced model.evaluate with model.predict

# This code cell identifies categorical features,
# so that the tabular models will work.
# This code cells does not have the path names variables, and does not have the
# save-benchmark_results and the save_error functions.
# These paths and variables are described in the Global Definitions cell above.

import pandas as pd
import os
import csv
import time
import gc
import torch
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, brier_score_loss
from pytorch_tabular import TabularModel
from pytorch_tabular.models import (
    AutoIntConfig, CategoryEmbeddingModelConfig, DANetConfig,
    FTTransformerConfig, GANDALFConfig, GatedAdditiveTreeEnsembleConfig,
    NodeConfig, TabNetModelConfig, TabTransformerConfig
)
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig
from torchinfo import summary
from sklearn.impute import SimpleImputer
import traceback

# ── Model list ────────────────────────────────────────────────────────────────
MODEL_CONFIGS = [
    ('AutoInt',                  AutoIntConfig),
    ('CategoryEmbedding',        CategoryEmbeddingModelConfig),
    ('DANet',                    DANetConfig),
    ('FTTransformer',            FTTransformerConfig),
    ('GANDALF',                  GANDALFConfig),
    ('GatedAdditiveTreeEnsemble',GatedAdditiveTreeEnsembleConfig),
    ('TabTransformer',           TabTransformerConfig),
    ('NODE',                     NodeConfig),
]


# ── Shared trainer config ─────────────────────────────────────────────────────
trainer_conf = TrainerConfig(
    batch_size=1024,     # 1024 is choice
    max_epochs=40,
    accelerator='gpu',
    devices=1,
    early_stopping='valid_loss',
    early_stopping_patience=5,
    early_stopping_min_delta=0.001,
    precision='16-mixed',
    progress_bar='none',
)

# ── Processor label ───────────────────────────────────────────────────────────
processor_type = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

# ══ Main benchmarking loop ════════════════════════════════════════════════════
for dataset_file in DATASETS:
    dataset_path = DATASET_DIR + dataset_file
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_file}")
    print(f"{'='*60}")

# ── Load and split dataset ────────────────────────────────────────────────
    try:
        df = pd.read_csv(dataset_path)
        from sklearn.model_selection import train_test_split
        from sklearn.preprocessing import LabelEncoder

        # Special chars in columnn names cause models to error out. Remove them.
        df.columns = df.columns.str.replace(r'[^\w]', '_', regex=True).str.strip('_')

        # Strip embedded quotes from string column values. Causes models to crash otherwise.
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].str.strip("'")

        X = df.drop(columns=['target'])
        y = LabelEncoder().fit_transform(df['target']).astype('int32')

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Detect categorical vs numeric columns
        cat_cols = [col for col in X.columns if X[col].dtype == 'object']
        num_cols = [col for col in X.columns if X[col].dtype != 'object']

        # Must impute NaNs in numeric columns only, else tabular models error out
        imputer = SimpleImputer(strategy='median')
        X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
        X_test[num_cols]  = imputer.transform(X_test[num_cols])

        # Convert to float32 — pytorch_tabular requires float, not int
        X_train[num_cols] = X_train[num_cols].astype('float32')
        X_test[num_cols]  = X_test[num_cols].astype('float32')

        n_samples              = len(df)
        n_features             = X_train.shape[1]
        n_numeric_features     = len(num_cols)
        n_categorical_features = len(cat_cols)

        feature_cols = list(X.columns)
        train_df = pd.DataFrame(X_train.values, columns=feature_cols)
        train_df['target'] = y_train
        test_df = pd.DataFrame(X_test.values, columns=feature_cols)
        test_df['target'] = y_test

        data_conf = DataConfig(
            target=['target'],
            continuous_cols=num_cols,
            categorical_cols=cat_cols,
            handle_missing_values=True
        )


    except Exception as e:
        print(f"Failed to load dataset {dataset_file}: {e}")
        save_error(dataset_file, 'DATA_LOAD', e)
        continue

    # ── Model loop ────────────────────────────────────────────────────────────
    for model_name, ModelConfigClass in MODEL_CONFIGS:
        print(f"\n  Model: {model_name} | Dataset: {dataset_file}")
        cell_start = time.time()

        try:
            # Reset GPU memory stats for clean peak measurement
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

            model_conf = ModelConfigClass(task='classification')

            model = TabularModel(
                data_config=data_conf,
                model_config=model_conf,
                optimizer_config=OptimizerConfig(),
                trainer_config=trainer_conf,
                verbose=0
            )

            # Training
            training_start = time.time()
            model.fit(train=train_df)
            training_time = time.time() - training_start

            # Capture epochs run and early stopping flag BEFORE deleting model
            n_epochs_run  = model.trainer.current_epoch
            early_stopped = n_epochs_run < trainer_conf.max_epochs

            # GPU peak memory
            gpu_memory_peak_mb = (
                torch.cuda.max_memory_allocated() / 1024**2
                if torch.cuda.is_available() else None
            )

            # Inference timing (using predict for throughput)
            inference_start = time.time()
            predictions     = model.predict(test_df)
            inference_time  = time.time() - inference_start

            pred_probas  = predictions['target_1_probability'].values
            pred_labels  = predictions['target_prediction'].values
            pred_sec     = len(test_df) / inference_time if inference_time > 0 else float('inf')

            # Calculate accuracy directly from predictions — avoids model.evaluate()
            # which triggers a torchmetrics multiclass bug on some datasets
            accuracy     = (pred_labels == test_df['target'].values).mean()

            # Metrics
            f1_weighted  = f1_score(test_df['target'], pred_labels, average='weighted')
            auc          = roc_auc_score(test_df['target'], pred_probas)
            brier        = brier_score_loss(test_df['target'], pred_probas)
            mean_pred_proba = pred_probas.mean()

            # Trainable parameters
            n_trainable_params = sum(
                p.numel() for p in model.model.parameters() if p.requires_grad
            )

            # FLOPs
            flops = None
            try:
                dummy_input = {'continuous_data': torch.randn(1, n_features).to('cuda')}
                model_summary = summary(model.model, input_data=(dummy_input,), verbose=0)
                flops = model_summary.total_mult_adds
            except Exception as flops_error:
                print(f"    FLOPs not calculated: {flops_error}")

            # Timing
            total_time             = time.time() - cell_start
            runtime_per_1000       = (total_time / n_samples) * 1000

            # Print summary
            print(f"    Accuracy:    {accuracy:.4%}")
            print(f"    F1:          {f1_weighted:.4f}")
            print(f"    AUC:         {auc:.4f}")
            print(f"    Brier:       {brier:.4f}")
            print(f"    Train time:  {training_time:.1f}s")
            print(f"    Epochs run:  {n_epochs_run} (early stopped: {early_stopped})")
            print(f"    Params:      {n_trainable_params:,}")
            print(f"    GPU peak:    {gpu_memory_peak_mb:.1f} MB" if gpu_memory_peak_mb else "    GPU peak: N/A")

            # Save results
            save_benchmark_results(
                dataset_name=dataset_file,
                n_samples=n_samples,
                n_features=n_features,
                n_numeric_features=n_numeric_features,
                n_categorical_features=n_categorical_features,
                model_name=model_name,
                accuracy=accuracy,
                f1_weighted=f1_weighted,
                auc=auc,
                brier_score=brier,
                training_time=training_time,
                inference_time=inference_time,
                pred_sec=pred_sec,
                total_time=total_time,
                runtime_per_1000_samples=runtime_per_1000,
                processor_type=processor_type,
                n_trainable_params=n_trainable_params,
                n_epochs_run=n_epochs_run,
                early_stopped=early_stopped,
                gpu_memory_peak_mb=gpu_memory_peak_mb,
                flops=flops,
                mean_pred_proba=mean_pred_proba,
                file_path=RESULTS_PATH
            )



        except Exception as e:
            print(f"    FAILED: {e}")
            save_error(dataset_file, model_name, e)

        finally:
            # Clean up GPU memory regardless of success or failure
            try:
                del model
            except:
                pass
            gc.collect()
            torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Benchmarking of the pytorch tabular models is now complete.")
print(f"{'='*60}")

Loop through four WideDeep models on multiple datasets

In [ ]:
# LOOP THROUGH FOUR WIDEDEEP DL MODELS ON SELECTED MULTIPLE DATASETS

# Includes these modifications:
# categorical column detection, NaN Imputation, DataFrame reconstruction, and Column name cleaning.

# Identifies categorical features in each dataset, which is information
# needed by the WideDeep models.

# TabMlp is included as WideDeep_TabMlp because it's the base Wide & Deep architecture
# Each model name is prefixed with WideDeep_ to distinguish them from same-named
# pytorch_tabular models in your results CSV
# The preprocessor is fit once per dataset and reused across all four models for consistency
# Early stopping uses pytorch_widedeep's native EarlyStopping callback


import pandas as pd
import numpy as np
import time
import gc
import torch
from sklearn.metrics import f1_score, roc_auc_score, brier_score_loss
from pytorch_widedeep import Trainer
from pytorch_widedeep.models import WideDeep, TabMlp, SelfAttentionMLP, TabFastFormer, SAINT
from pytorch_widedeep.preprocessing import TabPreprocessor
from pytorch_widedeep.metrics import Accuracy
from pytorch_widedeep.callbacks import EarlyStopping
from torchinfo import summary
from sklearn.impute import SimpleImputer

# pytorch_widedeep has additional logging that isn't controlled by verbose=0
# on the Trainer or TabPreprocessor. It comes from the underlying PyTorch Lightning
# trainer which logs epoch progress independently.
# This code suppresses it:
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

import warnings
warnings.filterwarnings('ignore', category=ResourceWarning)


# ── Model list ────────────────────────────────────────────────────────────────
WD_MODEL_CONFIGS = [
    ('WideDeep_TabMlp',        TabMlp),
    ('WideDeep_SelfAttnMLP',   SelfAttentionMLP),
    ('WideDeep_TabFastFormer', TabFastFormer),
    ('WideDeep_SAINT',         SAINT),
]

# ── Processor label ───────────────────────────────────────────────────────────
processor_type = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

# ── Early stopping callback ───────────────────────────────────────────────────
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta=0.001,
    restore_best_weights=True
)

# ══ Main benchmarking loop ════════════════════════════════════════════════════
for dataset_file in DATASETS:
    dataset_path = DATASET_DIR + dataset_file
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_file}")
    print(f"{'='*60}")

    # ── Load and split dataset ────────────────────────────────────────────────
    try:
        df = pd.read_csv(dataset_path)
        from sklearn.model_selection import train_test_split
        from sklearn.preprocessing import LabelEncoder

        # Special chars in column names cause models to error out. Remove them.
        df.columns = df.columns.str.replace(r'[^\w]', '_', regex=True).str.strip('_')
        # Strip embedded quotes from string column values
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].str.strip("'")

        X = df.drop(columns=['target'])
        y = LabelEncoder().fit_transform(df['target'])

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Detect categorical vs numeric columns
        cat_cols = [col for col in X.columns if X[col].dtype == 'object']
        num_cols = [col for col in X.columns if X[col].dtype != 'object']
        feature_cols = list(X.columns)

        # Must impute NaNs in numeric columns only, else models error out
        imputer = SimpleImputer(strategy='median')
        X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
        X_test[num_cols]  = imputer.transform(X_test[num_cols])

        n_samples              = len(df)
        n_features             = X.shape[1]
        n_numeric_features     = len(num_cols)
        n_categorical_features = len(cat_cols)

        # Preprocess once per dataset — reused across all models
        preprocessor = TabPreprocessor(
            cat_embed_cols=cat_cols if cat_cols else None,
            continuous_cols=num_cols if num_cols else None,
            cols_to_scale=num_cols if num_cols else None,
            verbose=0
        )

        X_train_df = X_train.copy() if isinstance(X_train, pd.DataFrame) else pd.DataFrame(X_train.values, columns=X.columns)
        X_test_df  = X_test.copy()  if isinstance(X_test,  pd.DataFrame) else pd.DataFrame(X_test.values,  columns=X.columns)

        X_train_wd = preprocessor.fit_transform(X_train_df)
        X_test_wd  = preprocessor.transform(X_test_df)

    except Exception as e:
        print(f"Failed to load dataset {dataset_file}: {e}")
        save_error(dataset_file, 'DATA_LOAD', e)
        continue

    # ── Model loop ────────────────────────────────────────────────────────────
    for model_name, ModelClass in WD_MODEL_CONFIGS:
        print(f"\n  Model: {model_name} | Dataset: {dataset_file}")
        cell_start = time.time()

        try:
            # Reset GPU memory stats
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

            # Build deep tabular component
            deep_model = ModelClass(
                column_idx=preprocessor.column_idx,
                continuous_cols=num_cols if num_cols else None      # this was =feature_cols (which did work)
            )
            model = WideDeep(deeptabular=deep_model)

            trainer = Trainer(
                model=model,
                objective='binary',
                metrics=[Accuracy()],
                callbacks=[EarlyStopping(
                    monitor='val_loss',
                    patience=5,
                    min_delta=0.001,
                    restore_best_weights=True
                )],
                verbose=0,
                precision='16-mixed'       # remove if error
            )

            # Training
            training_start = time.time()
            trainer.fit(
                X_tab=X_train_wd,
                target=y_train,
                n_epochs=40,
                batch_size=1024,          # 1024 is the preferred
                val_split=0.2
            )
            training_time = time.time() - training_start

            # Epochs run and early stopping
            n_epochs_run  = trainer.callback_container.callbacks[0].stopped_epoch \
                            if hasattr(trainer.callback_container.callbacks[0], 'stopped_epoch') \
                            else 40
            early_stopped = n_epochs_run < 40

            # GPU peak memory
            gpu_memory_peak_mb = (
                torch.cuda.max_memory_allocated() / 1024**2
                if torch.cuda.is_available() else None
            )

            # Inference timing
            inference_start = time.time()
            pred_probas = trainer.predict_proba(X_tab=X_test_wd)[:, 1]
            inference_time = time.time() - inference_start

            pred_labels = (pred_probas >= 0.5).astype(int)
            pred_sec    = len(X_test_wd) / inference_time if inference_time > 0 else float('inf')

            # Metrics
            accuracy        = (pred_labels == y_test).mean()
            f1_weighted     = f1_score(y_test, pred_labels, average='weighted')
            auc             = roc_auc_score(y_test, pred_probas)
            brier           = brier_score_loss(y_test, pred_probas)
            mean_pred_proba = pred_probas.mean()

            # Trainable parameters
            n_trainable_params = sum(
                p.numel() for p in model.parameters() if p.requires_grad
            )

            # FLOPs
            flops = None
            try:
                dummy_input = torch.randn(1, n_features).to(
                    'cuda' if torch.cuda.is_available() else 'cpu'
                )
                model_summary = summary(model.deeptabular, input_data=dummy_input, verbose=0)
                flops = model_summary.total_mult_adds
            except Exception as flops_error:
                print(f"    FLOPs not calculated: {flops_error}")

            # Timing
            total_time       = time.time() - cell_start
            runtime_per_1000 = (total_time / n_samples) * 1000

            # Print summary
            print(f"    Accuracy:    {accuracy:.4%}")
            print(f"    F1:          {f1_weighted:.4f}")
            print(f"    AUC:         {auc:.4f}")
            print(f"    Brier:       {brier:.4f}")
            print(f"    Train time:  {training_time:.1f}s")
            print(f"    Epochs run:  {n_epochs_run} (early stopped: {early_stopped})")
            print(f"    Params:      {n_trainable_params:,}")
            print(f"    GPU peak:    {gpu_memory_peak_mb:.1f} MB" if gpu_memory_peak_mb else "    GPU peak: N/A")

            # Save results
            save_benchmark_results(
                dataset_name=dataset_file,
                n_samples=n_samples,
                n_features=n_features,
                n_numeric_features=n_numeric_features,
                n_categorical_features=n_categorical_features,
                model_name=model_name,
                accuracy=accuracy,
                f1_weighted=f1_weighted,
                auc=auc,
                brier_score=brier,
                training_time=training_time,
                inference_time=inference_time,
                pred_sec=pred_sec,
                total_time=total_time,
                runtime_per_1000_samples=runtime_per_1000,
                processor_type=processor_type,
                n_trainable_params=n_trainable_params,
                n_epochs_run=n_epochs_run,
                early_stopped=early_stopped,
                gpu_memory_peak_mb=gpu_memory_peak_mb,
                flops=flops,
                mean_pred_proba=mean_pred_proba,
                file_path=RESULTS_PATH
            )

        except Exception as e:
            print(f"    FAILED: {e}")
            save_error(dataset_file, model_name, e)

        finally:
            try:
                del model
                del trainer
                del deep_model
            except:
                pass
            gc.collect()
            torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("The pytorch_widedeep benchmarking is now complete.")
print(f"{'='*60}")

The following code cells tune TabTransformer in an attempt to improve its accuracy.

In [ ]:
# -------------------------------------------
# SELECT DATASETS TO BE USED IN THE TRAINING/TESTING LOOP
# -------------------------------------------
# ── Select datasets by individual numbers and/or ranges ───────
# Examples: 42, "1-20", "81-105"

SELECTED = ["71-76","78-84","86-88","90-93","95-99"]

# ── Expand ranges into individual numbers ──────────────────────
def expand_selection(selected):
    result = []
    for item in selected:
        if isinstance(item, str) and '-' in item:
            start, end = item.split('-')
            result.extend(range(int(start), int(end) + 1))
        else:
            result.append(int(item))
    return result

selected_numbers = expand_selection(SELECTED)
DATASETS = [ALL_DATASETS[i-1] for i in selected_numbers]

# Confirm selection
print(f"Selected {len(DATASETS)} datasets:")
for i, d in zip(selected_numbers, DATASETS):
    print(f"  {i}. {d}")

Selected 25 datasets:
  71. in_vehicle_coupon_recommendation.csv
  72. insurance.csv
  73. Invistico_Airline_binarized_satisfied.csv
  74. ionosphere_converted_binarized_B.csv
  75. iranian_churn.csv
  76. iris_binarized.csv
  78. is_this_a_good_customer.csv
  79. jannis.csv
  80. jasmine_converted.csv
  81. jm1_binarized_TRUE.csv
  82. kc1_converted_binarized_TRUE.csv
  83. KDD_binarized_1.csv
  84. kdd_ipums_la_97_small_binarized_P.csv
  86. liver-disorders_converted_binarized_1.csv
  87. lupus_converted_binarized_DEAD.csv
  88. madeline_converted.csv
  90. make_blobs_dataset.csv
  91. make_circles_dataset.csv
  92. make_classification_binarized_4.csv
  93. make_moons_dataset.csv
  95. marketing_campaign_cleaned_dates_numeric.csv
  96. medical_appointment_no_shows_convert_dates_binarized_YES.csv
  97. Medical-Appointment.csv
  98. miniboone_binarized_TRUE.csv
  99. multiple_sclerosis_disease_cols_removed_binarized_1.csv


In [ ]:
# =============================================================================
# TABTRANSFORMER HYPERPARAMETER TUNING — UPDATED VERSION
# Changes: N_TRIALS=50, expanded search space, validation accuracy capture,
#          stratified splitting, dataset counter'=
#          Appends row to Excel file inside the loop, not at the conclusion
# =============================================================================

import warnings
import gc
import time
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from pytorch_tabular import TabularModel
from pytorch_tabular.models import TabTransformerConfig
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig
from pytorch_tabular.tabular_model_tuner import TabularModelTuner

# ── Tuning settings ────────────────────────────────────────────
N_TRIALS = 50
STRATEGY = 'random_search'

# ── Expanded search space ──────────────────────────────────────
search_space = {
    'model_config__num_heads':       [4, 8, 16],
    'model_config__num_attn_blocks': [2, 4, 6, 8],
    'model_config__input_embed_dim': [16, 32, 64, 128],
    'model_config__attn_dropout':    [0.0, 0.1, 0.2, 0.3],
    'model_config__ff_dropout':      [0.0, 0.1, 0.2, 0.3],
    'model_config__learning_rate':   [1e-4, 5e-4, 1e-3, 5e-3],
}

# ── Trainer config ─────────────────────────────────────────────
trainer_conf = TrainerConfig(
    batch_size=1024,
    max_epochs=40,
    accelerator='gpu',
    devices=1,
    early_stopping='valid_loss',
    early_stopping_patience=5,
    early_stopping_min_delta=0.001,
    precision='16-mixed',
    progress_bar='none',
)

# ── Results collector ──────────────────────────────────────────
tuning_results = []
total_datasets = len(DATASETS)

# ══ Main loop ══════════════════════════════════════════════════
for i, dataset_file in enumerate(DATASETS, start=1):
    dataset_path = DATASET_DIR / dataset_file
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_file}  ({i} of {total_datasets})")
    now_pacific = datetime.now(pytz.utc).astimezone(pacific_tz)
    print(f"Started at {now_pacific.strftime('%b %d, %Y %I:%M:%S %p')}")
    print(f"{'='*60}")

    dataset_start = time.time()

    try:
        # ── Load and preprocess ────────────────────────────────
        df = pd.read_csv(dataset_path)
        df.columns = df.columns.str.replace(r'[^\w]', '_', regex=True).str.strip('_')
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].str.strip("'")

        X = df.drop(columns=['target'])
        y = LabelEncoder().fit_transform(df['target']).astype('int32')

        # ── Stratified main split ──────────────────────────────
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y    # Stratifed to address imbalanced datasets
        )

        cat_cols = [col for col in X.columns if X[col].dtype == 'object']
        num_cols = [col for col in X.columns if X[col].dtype != 'object']

        imputer = SimpleImputer(strategy='median')
        X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
        X_test[num_cols]  = imputer.transform(X_test[num_cols])

        X_train[num_cols] = X_train[num_cols].astype('float32')
        X_test[num_cols]  = X_test[num_cols].astype('float32')

        feature_cols = list(X.columns)
        train_df = pd.DataFrame(X_train.values, columns=feature_cols)
        train_df['target'] = y_train

        test_full_df = pd.DataFrame(X_test.values, columns=feature_cols)
        test_full_df['target'] = y_test

        # ── Stratified val/test split ──────────────────────────
        val_df, test_df = train_test_split(
            test_full_df,
            test_size=0.5,
            random_state=42,
            stratify=test_full_df['target']
        )

        data_conf = DataConfig(
            target=['target'],
            continuous_cols=num_cols,
            categorical_cols=cat_cols,
            handle_missing_values=True
        )

        # ── Baseline accuracy (default hyperparameters) ────────
        print(f"  Running baseline (default hyperparameters)...")
        baseline_model = TabularModel(
            data_config=data_conf,
            model_config=TabTransformerConfig(task='classification'),
            optimizer_config=OptimizerConfig(),
            trainer_config=trainer_conf,
            verbose=0
        )
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            baseline_model.fit(train=train_df, validation=val_df)

        baseline_preds = baseline_model.predict(test_df)
        baseline_acc   = accuracy_score(
            test_df['target'],
            baseline_preds['target_prediction'].values
        )
        print(f"  Baseline accuracy: {baseline_acc:.4%}")

        del baseline_model
        gc.collect()
        torch.cuda.empty_cache()

        # ── Tuning ─────────────────────────────────────────────
        print(f"  Running {N_TRIALS} tuning trials...")
        tuner = TabularModelTuner(
            data_config=data_conf,
            model_config=TabTransformerConfig(task='classification'),
            optimizer_config=OptimizerConfig(),
            trainer_config=trainer_conf
        )

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            tuner_output = tuner.tune(
                train=train_df,
                validation=val_df,
                search_space=search_space,
                strategy=STRATEGY,
                n_trials=N_TRIALS,
                metric='accuracy',
                mode='max',
                progress_bar=False,
                verbose=False
            )

        # ── Extract results from OUTPUT object ─────────────────
        best_params  = tuner_output.best_params
        best_model   = tuner_output.best_model
        best_val_acc = tuner_output.best_score    # Best validation accuracy to later test for overfitting

        print(f"  Best params: {best_params}")

        # ── Evaluate best model on test set ────────────────────
        tuned_preds = best_model.predict(test_df)
        tuned_acc   = accuracy_score(
            test_df['target'],
            tuned_preds['target_prediction'].values
        )

        dataset_runtime = round(time.time() - dataset_start, 1)

        print(f"  Best val accuracy: {best_val_acc:.4%}")
        print(f"  Tuned accuracy:    {tuned_acc:.4%}")
        print(f"  Improvement:       {(tuned_acc - baseline_acc)*100:+.2f} pp")
        print(f"  Val/Test gap:      {(best_val_acc - tuned_acc)*100:+.2f} pp")
        print(f"  Runtime:           {dataset_runtime:.1f}s")

        tuning_results.append({
            'dataset':          dataset_file,
            'baseline_acc':     round(baseline_acc, 4),
            'tuned_acc':        round(tuned_acc, 4),
            'best_val_acc':     round(best_val_acc, 4),
            'val_test_gap':     round((best_val_acc - tuned_acc) * 100, 2),
            'improvement_pp':   round((tuned_acc - baseline_acc) * 100, 2),
            'n_trials':         N_TRIALS,
            'dataset_runtime':  dataset_runtime,
            'best_params':      str(best_params)
        })
        # ── Save results after each dataset ───────────────────────────
        output_file = Path('/content/drive/MyDrive/ml_dl_benchmark_results') / 'tabtransformer_tuning_results.xlsx'
        try:
            existing = pd.read_excel(output_file)
            pd.concat([existing, pd.DataFrame([tuning_results[-1]])], ignore_index=True).to_excel(output_file, index=False)
        except FileNotFoundError:
            pd.DataFrame([tuning_results[-1]]).to_excel(output_file, index=False)
        print(f"  Saved row to: {output_file}")

        del best_model
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        import traceback
        print(f"  FAILED: {e}")
        print(traceback.format_exc())
        tuning_results.append({
            'dataset':          dataset_file,
            'baseline_acc':     None,
            'tuned_acc':        None,
            'best_val_acc':     None,
            'val_test_gap':     None,
            'improvement_pp':   None,
            'n_trials':         N_TRIALS,
            'dataset_runtime':  round(time.time() - dataset_start, 1),
            'best_params':      str(e)
        })
        # ── Save results after each dataset ───────────────────────────
        output_file = Path('/content/drive/MyDrive/ml_dl_benchmark_results') / 'tabtransformer_tuning_results.xlsx'
        try:
            existing = pd.read_excel(output_file)
            pd.concat([existing, pd.DataFrame([tuning_results[-1]])], ignore_index=True).to_excel(output_file, index=False)
        except FileNotFoundError:
            pd.DataFrame([tuning_results[-1]]).to_excel(output_file, index=False)
        print(f"  Saved row to: {output_file}")

# ── Summary results ────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"TUNING SUMMARY")
print(f"{'='*60}")
df_results = pd.DataFrame(tuning_results)
print(df_results[['dataset', 'baseline_acc', 'tuned_acc',
                   'best_val_acc', 'val_test_gap',
                   'improvement_pp', 'dataset_runtime']].to_string(index=False))



  Best params: {'model_config__num_heads': 16, 'model_config__num_attn_blocks': 2, 'model_config__learning_rate': 0.005, 'model_config__input_embed_dim': 128, 'model_config__ff_dropout': 0.1, 'model_config__attn_dropout': 0.2, 'model': '0-TabTransformerConfig', 'loss_0': 0.580138087272644, 'loss': 0.580138087272644}
  Best val accuracy: 77.7778%
  Tuned accuracy:    82.1429%
  Improvement:       +35.71 pp
  Val/Test gap:      -4.37 pp
  Runtime:           95.8s
  Saved row to: /content/drive/MyDrive/ml_dl_benchmark_results/tabtransformer_tuning_results.xlsx

TUNING SUMMARY
                                                     dataset  baseline_acc  tuned_acc  best_val_acc  val_test_gap  improvement_pp  dataset_runtime
                        in_vehicle_coupon_recommendation.csv        0.7502     0.7549        0.7508         -0.41            0.47            819.0
                                               insurance.csv        0.7580     0.7580        0.7580          0.00            0